Sequential word count

In [32]:
import pyspark


def seq_word_count(filename: str) -> dict:
    counts = {}
    with open(filename) as f:
        for line in f:
            for word in line.split():
                counts[word] = counts[word]+1 if word in counts else 1
    return {k: v for k, v in sorted(counts.items(), key=lambda x: x[1], reverse=True)}


Running for the Hamlet text

In [50]:
filename = "../data/hamlet.txt"
topK = 10
result = seq_word_count(filename)
dict(list(result.items())[:topK])

{'the': 988,
 'and': 693,
 'of': 621,
 'to': 604,
 'I': 513,
 'a': 450,
 'my': 441,
 'in': 387,
 'HAMLET': 378,
 'you': 356}

Spark Version.
Setting the Spark session

In [46]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
             .master("local[*]") \
             .appName('word_count') \
             .getOrCreate()

Base RDD API version

In [49]:
def rdd_word_count(filename: str) -> pyspark.RDD:
    text_file = spark.sparkContext.textFile(filename)
    counts = text_file.flatMap(lambda line: line.split(" ")) \
        .map(lambda word: (word, 1)) \
        .reduceByKey(lambda x, y: x + y)
    return (counts.map(lambda i: (i[1], i[0]))
            .sortByKey(ascending=False)
            .map(lambda i: (i[1], i[0])))

result = rdd_word_count(filename)
result.take(topK)

[('the', 988),
 ('and', 693),
 ('of', 621),
 ('to', 604),
 ('I', 513),
 ('a', 450),
 ('my', 441),
 ('in', 387),
 ('HAMLET', 378),
 ('you', 356)]

PySpark SQL API version


In [53]:
from pyspark.sql.functions import split, explode, col

def sql_word_count(filename: str):
    text_file = spark.read.text(filename)
    words_df = text_file.withColumn("word", explode(split(col("value"), " ")))
    return words_df.groupBy("word").count().orderBy("count", ascending=False)

result = sql_word_count(filename)
result.show(topK)

DataFrame[value: string]
+------+-----+
|  word|count|
+------+-----+
|   the|  988|
|   and|  693|
|    of|  621|
|    to|  604|
|     I|  513|
|     a|  450|
|    my|  441|
|    in|  387|
|HAMLET|  378|
|   you|  356|
+------+-----+
only showing top 10 rows
